# PSTH Analysis with digitalin.dat Loading

## File Structure:
- `digitalin.dat` - Binary digital input file with TTL events
- `spikes.csv` - Spike times (same as before)
- Channel 0: Pico intervals
- Channel 1: Time markers

In [ ]:
# Import required modules
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import sys
sys.path.append('../..')

# Import custom modules
from digitalin_loader import load_digitalin_intervals, validate_intervals_compatibility
from psth_digitalin_analysis import run_psth_analysis_digitalin

# Reload the digitalin analysis module
if 'psth_digitalin_analysis' in sys.modules:
    importlib.reload(sys.modules['psth_digitalin_analysis'])
if 'digitalin_loader' in sys.modules:
    importlib.reload(sys.modules['digitalin_loader'])

In [ ]:
# Configuration - Update these paths to match your data location
data_folder = "../../../Data/052725_1"  # Adjust this path as needed
digitalin_file = os.path.join(data_folder, "digitalin.dat")
spikes_file = os.path.join(data_folder, "spikes.csv")

# Recording parameters
sampling_rate = 30000  # Hz - adjust if different
pico_channel = 0      # Channel for pico intervals
time_channel = 1      # Channel for time markers


In [ ]:
# PSTH Analysis Parameters - TEST SINGLE UNIT
units = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10 , 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]                    # Test with just Unit 1 first
duration = 25                  # Interval duration in ms (25ms, 10ms, 5ms)
bin_size_ms = 1.0              # Bin size in milliseconds
start_time = 1025000              # Start time for filtering in seconds (optional)
end_time =  1038000               # End time for filtering in seconds (optional)
max_trials = None               
pre_interval_ms = 5            # Milliseconds before interval to include
post_interval_ms = 10          # Milliseconds after interval to include
smooth_window = 3              # Number of bins for smoothing (None for no smoothing)
trial_ranges = None            # Specific trial ranges (None for all/max_trials)
save_plots = True             # Whether to save plots to file
output_path = "../../../Output/052725_1/PSTH_Clean"  # Directory to save plots

In [ ]:
# Run PSTH analysis with digitalin.dat
results = run_psth_analysis_digitalin(
            unit=units,  # Can be single unit or list
            duration=duration,
            bin_size_ms=bin_size_ms,
            start_time=start_time,
            end_time=end_time,
            max_trials=max_trials,
            pre_interval_ms=pre_interval_ms,
            post_interval_ms=post_interval_ms,
            smooth_window=smooth_window,
            trial_ranges=trial_ranges,
            spikes_file=spikes_file,
            digitalin_file=digitalin_file,
            sampling_rate=sampling_rate,
            use_digitalin=True, 
            save=save_plots,
            output_path=output_path
        )

# Display all plots (optional when saving)
if results:
    for i, (fig, axes) in enumerate(results):
        if fig:
            print(f"\n--- Unit {units[i]} ---")
            if not save_plots:  # Only show plots if not saving (to avoid cluttering)
                plt.show()
        else:
            print(f"No data found for Unit {units[i]}!")
else:
    print("No results generated!")

In [ ]:
# Test loading digitalin.dat file
if os.path.exists(digitalin_file):
    print("Loading digitalin.dat file...")
    
    try:
        intervals_df = load_digitalin_intervals(
            digitalin_filepath=digitalin_file,
            sampling_rate=sampling_rate,
            pico_channel=pico_channel,
            time_channel=time_channel
        )
        
        print(f"\nSuccessfully loaded {len(intervals_df)} intervals!")
        print("\nFirst few intervals:")
        print(intervals_df.head())
        
        # Validate compatibility
        is_valid = validate_intervals_compatibility(intervals_df)
        print(f"\nCompatibility check: {'✓ PASSED' if is_valid else '✗ FAILED'}")
        
    except Exception as e:
        print(f"Error loading digitalin.dat: {e}")
        intervals_df = None
        
else:
    print("digitalin.dat file not found! Please check the file path.")
    intervals_df = None